In [1]:
import json
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import MobileNetV2Config, MobileNetV2Model

2026-03-28 14:39:47.498318: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-28 14:39:47.508754: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774688987.520766 1057056 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774688987.524851 1057056 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774688987.535431 1057056 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [2]:
import sys
sys.path.append("/mnt/Personal/Projects/Autofocus/Code/Universal")

import pytorch_start # type: ignore
pytorch_start.activate_gpu(print_details=True)


                               ACTIVE GPU CONFIGURATION                              
+-------------+------------------------------------+---------------+--------------+---------------+
|   Device ID | Name                               |   Memory (GB) | PCI Bus ID   | GFX Version   |
+=============+====================================+===============+==============+===============+
|           0 | NVIDIA GeForce RTX 3050 Laptop GPU |          3.69 | Unknown      | Native        |
+-------------+------------------------------------+---------------+--------------+---------------+

Configured GPU 0: _CudaDeviceProperties(name='NVIDIA GeForce RTX 3050 Laptop GPU', major=8, minor=6, total_memory=3778MB, multi_processor_count=16, uuid=1fb0444c-042f-2836-2dde-93bbe9dca9b5, L2_cache_size=1MB)


In [3]:


dataset_type = "Test"
dataset_path = "/mnt/Velocity_Vault/Datasets/Autofocus/Dataset/"+dataset_type
model_path = "/mnt/Velocity_Vault/Datasets/Autofocus/Model/"

label_path = dataset_path+"/label.mm"
patch_path = dataset_path+"/patch.mm"

meta_data_path = dataset_path+"/meta.txt"

with open(meta_data_path, "r") as f:
    meta_data = json.load(f)

In [4]:
class TestDataset(Dataset):
    def __init__(self, patches_path, labels_path, shape):
        self.patches = np.memmap(patches_path, dtype=np.uint8, mode='r', shape=shape)
        self.labels = np.memmap(labels_path, dtype=np.uint8, mode='r', shape=(shape[0],))

        mask = (self.labels >= 1) & (self.labels <= 40)
        self.indices = np.where(mask)[0]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]

        x = torch.from_numpy(self.patches[real_idx])
        y = int(self.labels[real_idx]) - 1

        return x, torch.tensor(y, dtype=torch.long)

In [5]:
from transformers import MobileNetV2Config, MobileNetV2Model
import torch.nn as nn

class FocusNet(nn.Module):
    def __init__(self):
        super().__init__()

        config = MobileNetV2Config(
            num_channels=49,
            image_size=32,
            depth_multiplier=0.5,
            output_stride=8,
            finegrained_output=True
        )

        self.backbone = MobileNetV2Model(config)

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.head = nn.Sequential(
            nn.Linear(1280, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, 40)
        )

    def forward(self, x):
        # x: (B,49,32,32)

        outputs = self.backbone(x)
        feat = outputs.last_hidden_state   # (B, C, H, W)

        feat = self.pool(feat).flatten(1)

        out = self.head(feat)

        return out

In [6]:
def load_model(model_path, device):
    model = FocusNet().to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    return model

In [7]:
from tqdm import tqdm

def evaluate(model, loader, device, acc_k=2):
    model.eval()

    total_mae = 0.0
    total_acck = 0.0
    
    avg_mae, avg_acck = 0.0, 0.0

    class_range = torch.arange(40, device=device).float()

    pbar = tqdm(
        loader,
        desc="Test",
        leave=True,
        dynamic_ncols=True,
        smoothing=0.1
    )

    with torch.no_grad():
        for i, (x, y) in enumerate(pbar, 1):

            x = x.to(device, non_blocking=True).float().div(127.5).sub(1.0)
            y = y.to(device, non_blocking=True)

            outputs = model(x)

            prob = torch.softmax(outputs, dim=1)
            pred_cont = (prob * class_range).sum(dim=1)

            pred = pred_cont.round().long()

            # ---- metrics ----
            mae = (pred_cont - y).abs().mean()
            total_mae += mae.item()

            acck_val = ((pred - y).abs() <= acc_k).float().mean()
            total_acck += acck_val.item()

            # ---- running averages ----
            avg_mae = total_mae / i
            avg_acck = total_acck / i

            # ---- clean display ----
            pbar.set_postfix_str(
                f"mae={avg_mae:.3f} | acc@{acc_k}={avg_acck:.3f}"
            )

    print(f"\nTest Results:")
    print(f"MAE: {avg_mae:.4f}")
    print(f"Acc@{acc_k}: {avg_acck:.4f}")

    return avg_mae, avg_acck

In [8]:
# -----------------------------
# 5. Main
# -----------------------------
def run_test(
    patches_path,
    labels_path,
    shape,
    model_path,
    batch_size=512,
    acc_k=2
):
    device = "cuda"

    dataset = TestDataset(patches_path, labels_path, shape)

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    print(f"Testing on {len(dataset)} samples...")

    model = load_model(model_path, device)

    evaluate(model, loader, device, acc_k)

In [9]:
model_name = "good1.pth"

run_test(
    patches_path=patch_path,
    labels_path=label_path,
    shape=meta_data['patches']['shape'],
    model_path=model_path+"good1.pth",
    batch_size=1024
)


Testing on 55949 samples...


Test:   0%|          | 0/55 [00:00<?, ?it/s]/tmp/ipykernel_1057056/3154577441.py:15: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:203.)
  x = torch.from_numpy(self.patches[real_idx])
/tmp/ipykernel_1057056/3154577441.py:15: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_n


Test Results:
MAE: 2.3148
Acc@2: 0.7031


In [10]:

run_test(
    patches_path=patch_path,
    labels_path=label_path,
    shape=meta_data['patches']['shape'],
    model_path=model_path+"good2.pth",
    batch_size=1024
)

Testing on 55949 samples...


Test:   0%|          | 0/55 [00:00<?, ?it/s]/tmp/ipykernel_1057056/3154577441.py:15: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:203.)
  x = torch.from_numpy(self.patches[real_idx])
/tmp/ipykernel_1057056/3154577441.py:15: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_n


Test Results:
MAE: 2.6879
Acc@2: 0.6206


In [11]:

run_test(
    patches_path=patch_path,
    labels_path=label_path,
    shape=meta_data['patches']['shape'],
    model_path=model_path+"good3.pth",
    batch_size=1024
)

Testing on 55949 samples...


Test:   0%|          | 0/55 [00:00<?, ?it/s]/tmp/ipykernel_1057056/3154577441.py:15: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:203.)
  x = torch.from_numpy(self.patches[real_idx])
/tmp/ipykernel_1057056/3154577441.py:15: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_n


Test Results:
MAE: 2.4486
Acc@2: 0.6495
